In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os
import sys
from sklearn.preprocessing import StandardScaler # Mantener para uso directo/verificación potencial
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.nn import SmoothL1Loss

# Asegurar que la raíz del proyecto esté en el Python path
# Ajusta la profundidad del path ('../') según sea necesario
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
# Importar módulos refactorizados
from pampaneira_imputation import config
from pampaneira_imputation import data_loader as dl
from pampaneira_imputation import data_preprocessor as dp
from pampaneira_imputation import imputation_methods as im
from pampaneira_imputation import evaluation as ev
from pampaneira_imputation import utils


████████╗██╗███╗   ███╗███████╗    ███████╗███████╗██████╗ ██╗███████╗███████╗    █████╗ ██╗
╚══██╔══╝██║████╗ ████║██╔════╝    ██╔════╝██╔════╝██╔══██╗██║██╔════╝██╔════╝   ██╔══██╗██║
   ██║   ██║██╔████╔██║█████╗█████╗███████╗█████╗  ██████╔╝██║█████╗  ███████╗   ███████║██║
   ██║   ██║██║╚██╔╝██║██╔══╝╚════╝╚════██║██╔══╝  ██╔══██╗██║██╔══╝  ╚════██║   ██╔══██║██║
   ██║   ██║██║ ╚═╝ ██║███████╗    ███████║███████╗██║  ██║██║███████╗███████║██╗██║  ██║██║
   ╚═╝   ╚═╝╚═╝     ╚═╝╚══════╝    ╚══════╝╚══════╝╚═╝  ╚═╝╚═╝╚══════╝╚══════╝╚═╝╚═╝  ╚═╝╚═╝
ai4ts v0.0.3 - building AI for unified time-series analysis, https://time-series.ai 



In [3]:
# Cargar datos de intersección (filtrados para PAM_2)
try:
    df_int_raw = dl.load_intersection_data(
        filepath=config.INTERSECTION_FILE,
        target_truck_pos=config.TARGET_TRUCK_POS
    )
    print(f"Forma de los datos de intersección cargados: {df_int_raw.shape}")
    print(f"Rango de fechas: {df_int_raw[config.DATE_COL].min()} a {df_int_raw[config.DATE_COL].max()}")
    # display(df_int_raw.head()) # Opcional: mostrar encabezado
except Exception as e:
    print(f"Error al cargar los datos de intersección: {e}")
    # Detener ejecución o manejar el error apropiadamente
    raise

Forma de los datos de intersección cargados: (1335, 115)
Rango de fechas: 2023-01-17 17:00:00+00:00 a 2023-06-27 00:00:00+00:00


In [4]:
df_int_reindexed = dp.fill_missing_timestamps(df_int_raw, config.PERIOD_1_START, config.PERIOD_2_END)
print(f"Forma después de reindexar a periodos combinados: {df_int_reindexed.shape}")

Forma después de reindexar a periodos combinados: (3840, 114)


In [5]:
# Dividir en Periodos
periodo_1_raw, periodo_2_raw = dp.split_by_period(df_int_reindexed)

print(f"Forma del Periodo 1: {periodo_1_raw.shape}")
print(f"Forma del Periodo 2: {periodo_2_raw.shape}")

Forma del Periodo 1: (1332, 114)
Forma del Periodo 2: (480, 114)


In [9]:
import pandas as pd
import numpy as np

def calcular_nan_ratio(df: pd.DataFrame) -> dict:
    """
    Calcula el número de valores faltantes (NaN) y su proporción
    respecto al total de celdas en un DataFrame de pandas.

    Args:
        df (pd.DataFrame): El DataFrame de pandas a analizar.

    Returns:
        dict: Un diccionario con los siguientes resultados:
            - 'total_celdas': El número total de celdas en el DataFrame.
            - 'total_nan': El número total de valores NaN en el DataFrame.
            - 'nan_por_columna': Una serie de pandas con el conteo de NaN por columna.
            - 'proporcion_nan_total': La proporción de NaN sobre el total de celdas.
            - 'porcentaje_nan_total': El porcentaje de NaN sobre el total de celdas.
    """
    if not isinstance(df, pd.DataFrame):
        raise TypeError("La entrada debe ser un DataFrame de pandas.")

    # 1. Calcular el total de celdas en el DataFrame
    # df.shape devuelve una tupla (num_filas, num_columnas)
    total_filas = df.shape[0]
    total_columnas = df.shape[1]
    total_celdas = total_filas * total_columnas

    # 2. Calcular los valores NaN
    # df.isna() o df.isnull() devuelve un DataFrame booleano (True para NaN, False para valores existentes)
    # .sum() en un DataFrame booleano suma los True (considerados 1) por columna
    nan_por_columna = df.isna().sum()

    # Sumar el total de NaN de todas las columnas para obtener el total global
    total_nan = nan_por_columna.sum()

    # 3. Calcular la proporción y el porcentaje
    proporcion_nan_total = 0.0
    porcentaje_nan_total = 0.0

    if total_celdas > 0:
        proporcion_nan_total = total_nan / total_celdas
        porcentaje_nan_total = proporcion_nan_total * 100
    else:
        print("Advertencia: El DataFrame está vacío. La proporción de NaN será 0.")

    resultados = {
        'total_celdas': total_celdas,
        'total_nan': total_nan,
        'nan_por_columna': nan_por_columna,
        'proporcion_nan_total': proporcion_nan_total,
        'porcentaje_nan_total': porcentaje_nan_total
    }
    return resultados


In [12]:
periodo_1_raw.shape

(1332, 114)

In [13]:
periodo_2_raw.shape

(480, 114)

In [10]:
calcular_nan_ratio(periodo_1_raw)

{'total_celdas': 151848,
 'total_nan': np.int64(42894),
 'nan_por_columna': vehicles_PAM_1_OUT                           364
 vehicles_PAM_1_OUT_Zona_Granada              364
 vehicles_PAM_1_OUT_Zona_Catalunia_y_Otras    364
 vehicles_PAM_1_OUT_Zona_Andalucia_no_GR      364
 vehicles_PAM_1_OUT_Extranjero                364
                                             ... 
 TEMP                                         364
 RH                                           364
 WS                                           807
 WD                                           807
 PRES                                         364
 Length: 114, dtype: int64,
 'proporcion_nan_total': np.float64(0.28247984826932193),
 'porcentaje_nan_total': np.float64(28.247984826932193)}

In [11]:
calcular_nan_ratio(periodo_2_raw)

{'total_celdas': 54720,
 'total_nan': np.int64(15680),
 'nan_por_columna': vehicles_PAM_1_OUT                           135
 vehicles_PAM_1_OUT_Zona_Granada              135
 vehicles_PAM_1_OUT_Zona_Catalunia_y_Otras    135
 vehicles_PAM_1_OUT_Zona_Andalucia_no_GR      135
 vehicles_PAM_1_OUT_Extranjero                135
                                             ... 
 TEMP                                         135
 RH                                           135
 WS                                           148
 WD                                           148
 PRES                                         135
 Length: 114, dtype: int64,
 'proporcion_nan_total': np.float64(0.28654970760233917),
 'porcentaje_nan_total': np.float64(28.654970760233915)}